# autoplex_soap_turbo, from nothing to a matched spectrum

This notebook takes a new user through four things, in order:

1. **Installation** on every machine you use: the workstation you submit from, the cluster that runs DFT and turboGAP, and the cluster that runs `gap_fit`. It also sets up a **new MongoDB database** just for this tutorial.
2. **An iterative energy/force potential for Li–C–O** (Li₂CO₃ stoichiometry) that turboGAP can run. It uses autoplex's random-structure-search loop against VASP.
3. **A dipole model on top of it.** The Li–C–O potential is frozen and drives grand-canonical Monte Carlo over Li₂CO₃ clusters, and the dipole model is fitted iteratively against VASP.
4. **NMR matching with MADlib and LAMMPS.** An MD run is biased until its predicted NMR spectrum matches a target spectrum.

Parts 1–3 are jobflow flows. Their state and results live in your MongoDB database, so a run survives your laptop being closed. Part 4 is a LAMMPS run that you start by hand and that writes ordinary files.

**Running all the cells is safe.** Anything that touches a cluster, the queue or LAMMPS is off until you turn it on with the three switches below. Until then those cells print the command they would run. The cells that run here change nothing outside this repository. They build the workflow files and check them with `--dry-run` and `--stub`.

The references this notebook leans on, all in the repository:

| | |
|---|---|
| `README.md` | the reference, including the "Things that bite" table |
| `docs/training-guide.md` | the same path in prose, with LiF as the example |
| `docs/settings-reference.md` | every settings key, with its type and default |
| `docs/running.md` | watching, triaging and extracting from a submitted run |
| MADlib `README.md` | every `fix mad/*` and `compute mad/property/*` |

In [ ]:
RUN_SETUP = False   # Part 1: installs on every machine and starts the runner
SUBMIT = False      # Parts 2-3: puts flows in the queue
RUN_LAMMPS = False  # Part 4: needs an lmp built with MADLIB

In [ ]:
import getpass
import os
import subprocess
import textwrap
from pathlib import Path

# This notebook lives in <repo>/tutorial/.
REPO = Path.cwd().resolve().parent
PY = REPO / "autoplex_venv" / "bin" / "python"


def sh(cmd, act=True, cwd=REPO):
    """Run a bash command, stream its output, raise on a non-zero exit."""
    if not act:
        print("[not run]", cmd)
        return
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")


def write(path, text, **subs):
    """Write a file under the repository, filling {{NAME}} placeholders."""
    for key, value in subs.items():
        text = text.replace("{{" + key + "}}", str(value))
    if "{{" in text:
        raise ValueError(f"unfilled placeholder in {path}")
    path = REPO / path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip())
    print("wrote", path.relative_to(REPO))
    return path


print("repository:", REPO)

---
# Part 1: Installation

## 1.1 What runs where

One run is spread over several machines that **share no filesystem**:

```
 workstation / runner            cluster A (Roihu)             cluster B (Triton)
 ┌────────────────────┐   ssh    ┌──────────────────────┐      ┌──────────────────┐
 │ jf runner           │ ───────▶ │ VASP, turboGAP        │      │ gap_fit (QUIP)   │
 │ builds + submits    │ ───────────────────────────────────▶ │                  │
 └─────────┬──────────┘          └──────────┬───────────┘      └────────┬─────────┘
           │                                │  structures, potentials   │
           └──────────────────────▶  MongoDB  ◀─────────────────────────┘
                                 (queue + outputs + GridFS)
```

* **MongoDB** holds the queue and every output. Structures and potentials travel between stages *through the database*, never as file paths.
* **The runner** is the one machine with `roles=runner`. It opens ssh connections to the others, submits Slurm jobs and moves results. The `host` of every other machine must be reachable over ssh *from the runner*.
* **Roles** decide what is installed where. `vasp`, `aims`, `turbogap`, `gapfit` and `runner` each generate a jobflow-remote worker named `<machine>_<role>`, with dashes turned into underscores. `roihu-cpu` with `roles=vasp` gives the worker `roihu_cpu_vasp`, and the settings files name stages by these workers.

You edit **one file**, `config/machines.conf`, and everything else is generated from it. In this tutorial it is `config/machines.tutorial.conf`, so the tutorial gets its own database without touching an existing setup.

## 1.2 Before you start

* **Passwordless ssh** from the runner to every machine. Name a specific login node (`roihuc1`), not a round-robin alias.
* **CSC certificates last 24 hours.** Roihu needs a key plus a certificate signed in a browser. Renew it on your workstation and push it to the runner with `bash setup/refresh_csc_cert.sh`. When it lapses nothing reports an error. Jobs just stop advancing.
* **A `gap_fit` that can fit dipoles** for Part 3. `gap_fit dipole_parameter_name=` is not in stock QUIP. If you have such a build, point `gap_fit_env=` at a shell file that puts it on `PATH`. Otherwise `setup/build_quip.sh` builds stock QUIP, which is enough for Part 2 only.
* **A MongoDB database and a user that owns it.** Whoever administers your MongoDB creates them, for example in `mongosh`:

```js
use lico_tutorial
db.createUser({user: "you", pwd: "...", roles: [{role: "dbOwner", db: "lico_tutorial"}]})
```

Use a **fresh** database. `jf admin reset` below wipes the jobflow collections it finds, and the generated store writes to collections called `queue`, `outputs` and `outputs_blobs`. A database another tool already writes `outputs` into will get mixed up with this one.

## 1.3 Clone, and build the local environment

Clone with `--recursive`. `autoplex/` is a submodule pinned to the fork that teaches autoplex to fit `soap_turbo`.

```bash
git clone --recursive https://github.com/TiganyZ/autoplex_soap_turbo.git
cd autoplex_soap_turbo
bash setup/setup_env_uv.sh --work-dir "$PWD" --python 3.11 --roles runner
./autoplex_venv/bin/python -m ipykernel install --user --name autoplex_venv
jupyter lab tutorial/autoplex_soap_turbo_tutorial.ipynb
```

Then **switch this notebook's kernel to `autoplex_venv`**. The cells from 1.5 onwards import pymongo, ASE, jobflow-remote and this package.

On an Aalto workstation, keep the checkout on local disk, not in `$HOME`. A Python environment is about 28,000 files, and home volumes cap inodes. The install then dies with "Disk quota exceeded" while `df` shows space free.

In [ ]:
sh(f"{PY} -c 'import autoplex, autoplex_soap_turbo, jobflow_remote; print(\"environment OK\")'")

## 1.4 Your database and your machines

Fill these in. The password is never written into the notebook. It is read from `$TUTORIAL_MONGO_PASSWORD`, or asked for once, and goes only into the generated configuration files, which are owner-readable and gitignored.

For testing, `database` is set to `atomate2`. Your MongoDB user has to be authorised on that database. Being able to read some other database is not enough. 1.5 checks this before anything is installed.

The machines below are a real three-cluster layout: an Aalto workstation as runner, Roihu for VASP and turboGAP, and Triton for `gap_fit`. Change the hosts, accounts and directories to yours. The `work_dir`s are new directories, so the tutorial gets its own environments and does not upgrade an existing installation underneath a running campaign.

In [ ]:
PROJECT = "lico_tutorial"   # jobflow-remote project name; one project = one database

MONGO = {
    "host": "CHANGEME",
    "port": 27017,
    "database": "atomate2",
    "username": "CHANGEME",
    "auth_source": "",       # empty: authenticate against `database`
}

MACHINES = [
    {"name": "alto", "host": "alt", "roles": "runner",
     "work_dir": "/l/zarrout1/lico_tutorial", "python_version": "3.11"},
    {"name": "roihu-cpu", "host": "roihuc1", "roles": "vasp,turbogap",
     "work_dir": "/scratch/project_2003566/zarrout1/lico_tutorial",
     "scheduler": "slurm", "account": "project_2003566", "partition": "medium",
     "python_module": "python-data/3.12",
     "modules": "gcc/15.2.0,openmpi/5.0.10,openblas/0.3.30,netlib-scalapack/2.2.2",
     "max_jobs": 100, "vasp_cmd": "srun vasp_std"},
    {"name": "triton", "host": "tr", "roles": "gapfit",
     "work_dir": "/scratch/work/zarrout1/lico_tutorial",
     "scheduler": "slurm", "account": "sumo", "partition": "sumo",
     "python_module": "mamba", "modules": "triton/2025.1-gcc,gcc/13.3.0",
     "max_jobs": 10, "gap_fit_env": "/scratch/work/zarrout1/reece/dipole/test/env.sh"},
]

# Slurm requests used in Parts 2-3. jobflow-remote *replaces* a worker's
# resources with a stage's rather than merging them, so account and partition
# are repeated in every request.
ROIHU = {"account": "project_2003566"}
TRITON = {"account": "sumo", "partition": "sumo"}

MONGO_READY = "CHANGEME" not in (MONGO["host"], MONGO["username"])
MONGO_PASSWORD = os.environ.get("TUTORIAL_MONGO_PASSWORD", "")
if MONGO_READY and not MONGO_PASSWORD:
    MONGO_PASSWORD = getpass.getpass("MongoDB password: ")


def worker(role):
    """The jobflow-remote worker of the machine with this role."""
    for m in MACHINES:
        if role in m["roles"].split(","):
            return f"{m['name'].replace('-', '_')}_{role}"
    raise KeyError(f"no machine has role {role}")


def machine(role):
    return next(m for m in MACHINES if role in m["roles"].split(","))


print("workers:", [worker(r) for r in ("runner", "vasp", "turbogap", "gapfit")])

## 1.5 Can you reach the database?

Check this first, because every later failure that comes from it looks like something else. A wrong password or a missing role shows up here as `OperationFailure ... not authorized on <database>`, and not as a runner that never picks anything up.

In [ ]:
def check_mongo(mongo, password):
    import pymongo

    client = pymongo.MongoClient(
        host=mongo["host"], port=mongo["port"],
        username=mongo["username"], password=password,
        authSource=mongo["auth_source"] or mongo["database"],
        serverSelectionTimeoutMS=5000,
    )
    db = client[mongo["database"]]
    db.command("ping")
    probe = db["tutorial_write_probe"]
    probe.insert_one({"probe": True})
    probe.drop()
    print(f"read/write OK on '{mongo['database']}'; existing collections:",
          sorted(db.list_collection_names()))


if MONGO_READY:
    check_mongo(MONGO, MONGO_PASSWORD)
else:
    print("[not run] fill in MONGO above first")

## 1.6 Write the machine configuration

This writes the same format as `config/machines.conf.example`: the MongoDB block, then one `machine` line per machine. The file holds the password, so it is created owner-readable only and matched by `.gitignore`.

Every key you can put on a machine line is in `docs/training-guide.md` §2. Two of them skip a build. `gap_fit_env=` points at an existing `gap_fit`, and `turbogap_bin=` at an existing turboGAP.

In [ ]:
import shlex

CONF = REPO / "config" / "machines.tutorial.conf"


def machines_conf(project, mongo, password, machines):
    """The text of a machines.conf."""
    lines = [
        f"project_name={project}",
        f"mongodb_host={mongo['host']}",
        f"mongodb_port={mongo['port']}",
        f"mongodb_database={mongo['database']}",
        f"mongodb_username={shlex.quote(mongo['username'])}",
        f"mongodb_password={shlex.quote(password)}",
        f"mongodb_auth_source={mongo['auth_source']}",
        "",
    ]
    for m in machines:
        keys = [f"{k}={shlex.quote(str(v))}" for k, v in m.items() if k != "name"]
        lines.append(f"machine {m['name']} \\\n    " + " \\\n    ".join(keys))
    return "\n".join(lines) + "\n"


def write_conf(path, text):
    path.touch(mode=0o600, exist_ok=True)
    path.chmod(0o600)
    path.write_text(text)
    print("wrote", path)


if MONGO_READY:
    write_conf(CONF, machines_conf(PROJECT, MONGO, MONGO_PASSWORD, MACHINES))
else:
    print(machines_conf(PROJECT, MONGO, "<password>", MACHINES))
sh(f"git check-ignore -q {CONF} && echo 'gitignored: yes'")

## 1.7 Install on every machine

`setup_all_machines.sh` goes through every declared machine and, on each one:

1. rsyncs this repository (the working tree, submodule included) to `<work_dir>/autoplex_soap_turbo`,
2. builds a `uv` environment. autoplex goes only on machines with `runner`, `gapfit` or `turbogap`, because a VASP-only machine never imports it,
3. builds turboGAP on `turbogap` machines, and QUIP on `gapfit` machines that have no `gap_fit_env`,
4. installs `<work_dir>/env.sh`, which every worker sources,

and then renders the jobflow-remote project and installs it on the runner as `~/.jfremote/<project>.yaml`.

It is safe to re-run. `--only <machine>` limits it to one machine. `DRY_RUN=1` prints what it would do. The first full run takes tens of minutes, and most of that is the turboGAP build.

Rendering writes into `config/generated/`, which is shared with your main configuration. After the tutorial, a plain `bash setup/setup_all_machines.sh --config-only` puts your main project's files back.

In [ ]:
sh(f"DRY_RUN=1 bash setup/setup_all_machines.sh --config {CONF}", act=MONGO_READY)

In [ ]:
sh(f"bash setup/setup_all_machines.sh --config {CONF}", act=RUN_SETUP)
sh(f"bash setup/check_setup.sh --config {CONF}", act=RUN_SETUP)

## 1.8 Start the runner

On the runner, `jf admin reset` initialises the new database and `jf runner start` starts the daemon that moves jobs along. `-p` names the project explicitly, so this cannot act on another project the runner already has.

**`jf admin reset -f` deletes every flow in the project's queue without asking.** On a fresh database that is what you want. Never point it at a database that holds someone else's work.

The project file is also installed locally, so this notebook can submit flows and read results. Both only need MongoDB.

In [ ]:
RUNNER = machine("runner")
on_runner = f"ssh {RUNNER['host']} 'source {RUNNER['work_dir']}/env.sh && "

sh(on_runner + f"jf -p {PROJECT} admin reset -f && jf -p {PROJECT} runner start'", act=RUN_SETUP)
sh(on_runner + f"jf -p {PROJECT} project check --errors'", act=RUN_SETUP)
sh(f"mkdir -p ~/.jfremote && install -m 600 config/generated/jobflow_remote.yaml ~/.jfremote/{PROJECT}.yaml",
   act=RUN_SETUP)

## 1.9 A first run that needs none of it

`examples/minimal/` is a complete iterative run. With `--stub` the fit, the sampler and the DFT are replaced by functions that return outputs of the same shape, and everything between them runs for real in an in-memory store. That covers payload routing, every serialisation boundary and the convergence logic, with no cluster, no binaries and no database. It takes seconds. If it fails, the installation is broken, and nothing later will work.

In [ ]:
sh(f"{PY} examples/minimal/make_seed.py")
sh(f"{PY} examples/minimal/run.py --stub 2>&1 | tail -n 15")

---
# Part 2: An iterative Li–C–O potential for turboGAP

## 2.1 Which workflow

An **energy/force** model for bulk, periodic Li–C–O comes from `workflows/vasp_iterative/`. That is autoplex's random-structure-search (RSS) loop, with VASP statics as the reference:

```
buildcell random cells ─▶ select (CUR) ─▶ VASP statics ─▶ gap_fit ─▶ RSS relax with that GAP ─▶ select ─▶ VASP ─▶ fit ─▶ …
```

Each round relaxes new random cells **with the model just fitted** and sends the most informative ones back to DFT. The model therefore learns where it is wrong. The fit uses only the descriptors turboGAP evaluates natively: `soap_turbo`, `distance_2b` and `angle_3b`. That is what makes the result loadable by turboGAP, and therefore usable as the sampler in Part 3.

Two files configure it. `rss_config.yaml` is autoplex's own `RssConfig`, which covers the search, the INCAR and the hyperparameters. `vasp_rss.yaml` is this repository's file and covers which worker runs what, with which resources.

## 2.2 The search: `rss_config.yaml`

The choices that are specific to Li–C–O are:

* **`SPECIES Li%NUM=2,C%NUM=1,O%NUM=3`, `NFORM {1,2,3,4}`**: every random cell is a whole number of Li₂CO₃ formula units, so it is neutral and stoichiometric by construction, with 6–24 atoms.
* **`MINSEP`**: minimum separations, as ranges buildcell samples from. They are set below the bonds of crystalline Li₂CO₃ (C–O 1.28–1.30 Å, Li–O 1.9–2.0 Å) so that buildcell does not place atoms on top of one another. Note that buildcell places atoms, not carbonate ions. Early rounds will contain C–C and O–O bonds that Li₂CO₃ does not have. That is part of what RSS is for, but it costs DFT. If you only care about the carbonate phase, seed it with known structures as well.
* **`hookean_paras`**: a repulsive floor per pair of atomic numbers (Li 3, C 6, O 8), as `[force constant, threshold]`. Each threshold sits below the shortest real bond, so it stops collapse without enforcing a bond.
* **`ENCUT 520`, `ISMEAR 0`, `SIGMA 0.05`**: O and the Li_sv PAW are converged at 520 eV. Random cells can be metallic, so the smearing is small but not tetrahedron.
* **The descriptor block**: `cutoff_transition_width: 0.5` for `distance_2b` and `angle_3b` is **required**. turboGAP hardcodes that buffer, and any other value fits and converts without complaint and then disagrees with QUIP. `angle_3b` cannot use `sparse_method: uniform`. `alpha_max` above 7 needs `poly3gauss`.

Every value is a starting point, chosen to make the loop run. It has not been converged for Li₂CO₃.

In [ ]:
RSS_CONFIG = r"""
# autoplex RssConfig for Li2CO3-stoichiometry Li-C-O, fitting a turboGAP-compatible
# energy/force GAP. Every key not set here falls back to autoplex.settings.RssConfig.
tag: LiCO
train_from_scratch: true

generated_struct_numbers: [200]

buildcell_options:
  - ABFIX: false
    NFORM: "{1,2,3,4}"
    SYMMOPS: 1-4
    OCTET: false
    SPECIES: Li%NUM=2,C%NUM=1,O%NUM=3
    MINSEP: 1.0 C-O=1.15-1.45 Li-O=1.7-2.3 Li-C=2.3-3.2 Li-Li=2.2-3.2 O-O=2.0-2.8 C-C=2.4-4.0

num_processes_buildcell: 32
num_of_initial_selected_structs: [20]
num_of_rss_selected_structs: 20
initial_selection_enabled: true
rss_selection_method: bcur2i

bcur_params:
  soap_paras:
    l_max: 8
    n_max: 8
    atom_sigma: 0.5
    cutoff: 5.5
    cutoff_transition_width: 0.5
    zeta: 4.0
    average: true
    species: true
  frac_of_bcur: 0.8
  bolt_max_num: 300

random_seed: 42

# Isolated atoms give the e0 references; Part 3 quotes its GCMC chemical
# potential against them. Dimers pin the short-range repulsion.
include_isolated_atom: true
isolatedatom_box: [20.0, 20.0, 20.0]
include_dimer: true
dimer_box: [20.0, 20.0, 20.0]
dimer_range: [0.9, 6.0]
dimer_num: 21

custom_incar:
  ENCUT: 520.0
  ISMEAR: 0
  SIGMA: 0.05
  PREC: Accurate
  ADDGRID: .TRUE.
  EDIFF: 1e-07
  NELM: 250
  LWAVE: .FALSE.
  LCHARG: .FALSE.
  ALGO: normal
  ISYM: 0
  KSPACING: 0.25
  LREAL: .FALSE.
  KPAR: 4
  NCORE: 8

dft_ref_file: vasp_ref.extxyz
config_types: [initial, traj_early, traj]
rss_group: [traj]
test_ratio: 0.1

# Each structure's sigma follows its energy above the convex hull.
regularization: true
scheme: linear-hull

mlip_type: GAP
ref_energy_name: REF_energy
ref_force_name: REF_forces
ref_virial_name: REF_virial
auto_delta: true
num_processes_fit: 48

max_iteration_number: 5
stop_criterion: 0.001
num_processes_rss: 32
device_for_rss: cpu

# [force constant, threshold] per atomic-number pair: Li 3, C 6, O 8.
hookean_repul: true
hookean_paras:
  "(3, 3)": [1000, 1.8]
  "(3, 6)": [1000, 1.8]
  "(3, 8)": [1000, 1.4]
  "(6, 6)": [1000, 1.1]
  "(6, 8)": [1000, 0.9]
  "(8, 8)": [1000, 1.0]

mlip_hypers:
  GAP:
    general:
      two_body: false
      three_body: false
      soap: false
      distance_2b: true
      angle_3b: true
      soap_turbo: true
      default_sigma: "{0.0001 0.05 0.05 0}"
      sparse_jitter: 1.0e-8
      openmp_chunk_size: 1000
    distance_2b:
      cutoff: 5.5
      cutoff_transition_width: 0.5
      n_sparse: 40
      covariance_type: ard_se
      delta: 0.5
      theta_uniform: 0.5
      sparse_method: uniform
    angle_3b:
      cutoff: 3.25
      cutoff_transition_width: 0.5
      n_sparse: 100
      covariance_type: ard_se
      delta: 0.5
      theta_uniform: 1.0
      sparse_method: cur_points
    soap_turbo:
      rcut_hard: 5.5
      rcut_soft: 5.0
      l_max: 8
      alpha_max: 8
      basis: poly3gauss
      atom_sigma_r: 0.5
      atom_sigma_t: 0.5
      atom_sigma_r_scaling: 0.0
      atom_sigma_t_scaling: 0.0
      amplitude_scaling: 1.0
      central_weight: 1.0
      compress_soap: "true"
      compress_mode: trivial
      zeta: 4
      covariance_type: dot_product
      delta: 0.1
      f0: 0.0
      n_sparse: 500
      sparse_method: cur_points
"""
write("workflows/lico/rss_config.yaml", RSS_CONFIG);

## 2.3 Where it runs: `vasp_rss.yaml`

Three groups of jobs, three workers. VASP statics go to the VASP machine with the `roihu_vasp_config` environment from `config/exec_configs.yaml`. `gap_fit` goes to the fit machine. Structure generation and selection need `buildcell`, which ships with the Python environment, so they go to the general worker on the VASP machine.

In [ ]:
VASP_RSS = r"""
# Li-C-O energy/force potential: autoplex RSS against VASP.
#     python workflows/vasp_iterative/run.py --config workflows/lico/vasp_rss.yaml
name: lico_rss
# No `project:` here; run.py takes --project.
rss_config_file: rss_config.yaml

vasp:
  worker: {{VASP_WORKER}}
  exec_config: roihu_vasp_config
  resources: {account: {{ROIHU_ACCOUNT}}, partition: medium, nodes: 1,
              ntasks_per_node: 128, time: "02:00:00"}

fit:
  worker: {{FIT_WORKER}}
  resources: {account: {{TRITON_ACCOUNT}}, partition: {{TRITON_PARTITION}}, nodes: 1,
              cpus_per_task: 48, time: "1-00:00:00"}

general:
  worker: {{GENERAL_WORKER}}
  resources: {account: {{ROIHU_ACCOUNT}}, partition: small, nodes: 1,
              ntasks_per_node: 32, time: "04:00:00"}

# Baseline for INCAR keys rss_config.yaml does not set; rss_config.yaml wins.
user_incar_settings: {}
isolated_atom_kspacing: 100.0
"""
write("workflows/lico/vasp_rss.yaml", VASP_RSS,
      VASP_WORKER=worker("vasp"),
      FIT_WORKER=worker("gapfit"),
      GENERAL_WORKER=machine("vasp")["name"].replace("-", "_") + "_worker",
      ROIHU_ACCOUNT=ROIHU["account"],
      TRITON_ACCOUNT=TRITON["account"], TRITON_PARTITION=TRITON["partition"]);

## 2.4 Check it, then submit

`--dry-run` builds the whole flow and prints every job with its worker, plus the INCAR as VASP will receive it, without submitting anything. Read the worker column. A stage on the wrong machine is the mistake a dry run catches.

Before you submit after any code change, run `bash setup/setup_all_machines.sh --config config/machines.tutorial.conf --sync-only`. Each machine imports from its **own** copy of the repository. If only the runner's copy is updated, the flow builds and submits fine and then dies on the first stage that lands on another cluster.

In [ ]:
RSS_RUN = f"{PY} workflows/vasp_iterative/run.py --config workflows/lico/vasp_rss.yaml"
sh(RSS_RUN + " --dry-run")

In [ ]:
sh(f"bash setup/setup_all_machines.sh --config {CONF} --sync-only", act=SUBMIT)
sh(RSS_RUN + f" --project {PROJECT}", act=SUBMIT)

## 2.5 Watching it

From anywhere that has the project file:

```bash
jf -p lico_tutorial flow list                 # flows and their ids
jf -p lico_tutorial job list -fid <flow-id>   # every job's state
jf -p lico_tutorial job info <db-id>          # including the traceback, if it failed
jf -p lico_tutorial job rerun -did <db-id>    # after fixing and --sync-only
```

`docs/running.md` covers triage. The most common silent stall on CSC machines is an expired certificate: jobs sit in `CHECKED_OUT` or `UPLOADED` and the logs say nothing.

In [ ]:
sh(f"{REPO}/autoplex_venv/bin/jf -p {PROJECT} flow list", act=SUBMIT)

## 2.6 Getting the potential out, and converting it once

autoplex's RSS fit writes its GAP on the fit machine and records the path as `mlip_path` in the job's output. The cell below finds the most recent one in **your** database.

The conversion to turboGAP's `.gap` format is done **once**, on the machine that will run turboGAP, where `make_gap_files.py` is present. Part 3 then uses that `.gap` directly. Do not point Part 3 at the XML. That reconverts every iteration, and the conversion drops any descriptor with an empty sparse set, `core_pot` among them. A LiF potential that lost its core repulsion this way let relaxations collapse to 0.39 Å bonds, which no DFT code would run. After converting, check which `gap_beg` blocks survived.

In [ ]:
def latest_mlip_path(project):
    """The newest `mlip_path` any job in this project's output store recorded."""
    from jobflow_remote import get_jobstore

    store = get_jobstore(project_name=project)
    store.connect()
    doc = store.query_one({"output.mlip_path": {"$exists": True}},
                          properties=["name", "output.mlip_path", "completed_at"],
                          sort={"completed_at": -1})
    path = doc["output"]["mlip_path"]
    path = path[0] if isinstance(path, list) else path
    print(f"{doc['name']}: {path}")
    return str(path)


SAMPLER = machine("turbogap")
POTENTIAL_DIR = f"{SAMPLER['work_dir']}/lico_potential"
LICO_GAP = f"{POTENTIAL_DIR}/gap_files/LiCO/LiCO.gap"

CONVERT = """
from autoplex_soap_turbo.turbogap.potential import convert_potential
import glob
xml = sorted(glob.glob('xml/*.xml'))[0]
gap, dropped = convert_potential(xml, ['Li', 'C', 'O'], '.', 'LiCO')
print(gap, 'dropped:', dropped)
"""

if SUBMIT:
    fit_dir = Path(latest_mlip_path(PROJECT))
    fit_host = machine("gapfit")["host"]
    sh(f"ssh {SAMPLER['host']} mkdir -p {POTENTIAL_DIR}/xml")
    # .xml and its .sparseX files travel together.
    sh(f"scp -3 '{fit_host}:{fit_dir}/*.xml*' '{SAMPLER['host']}:{POTENTIAL_DIR}/xml/'")
    sh(f"ssh {SAMPLER['host']} 'source {SAMPLER['work_dir']}/env.sh && cd {POTENTIAL_DIR} "
       f"&& python -c {shlex.quote(CONVERT)} && grep ^gap_beg {LICO_GAP} | sort | uniq -c'")
else:
    print("[not run] the converted potential will be", f"{SAMPLER['host']}:{LICO_GAP}")

---
# Part 3: A dipole model on top of the potential

## 3.1 The loop, and "Mode B"

```
fit dipole ─▶ GCMC with the frozen LiCO.gap ─▶ select (FPS) ─▶ VASP: dipole + polarizability ─▶ merge ─▶ fit ─▶ …
```

The dipole workflow fits a GAP whose gradient is a dipole, using `soap_turbo` and `gap_fit dipole_parameter_name=`. A dipole model has no energy and no forces, so it cannot drive sampling itself. Here the **frozen** Li–C–O potential from Part 2 drives the walk, and only the dipole model iterates. This is "Mode B" in `docs/training-guide.md`. The dipole model fitted in each iteration rides along in the same turboGAP run, with its blocks flagged `dipole_model = .true.`. Every sampled frame therefore carries the model's own prediction, which shows where it is extrapolating. Those predictions are stripped before the frame goes to DFT, so the model can never be trained on its own answers.

Three physical constraints shape the settings:

* **Isolated, neutral clusters.** A total dipole is defined only for a non-periodic system, and only origin-independently for a neutral one. So `periodic: false`, VASP runs each cluster `molecular: true` in a box, and the walk exchanges **whole Li₂CO₃ units**, never single ions. A lone Li⁺ insertion would make a charged cluster that the VASP stage refuses after the DFT has already been queued.
* **VASP dipoles.** `LCALCPOL` gives the Berry-phase dipole and `LEPSILON` the dielectric tensor that the polarizability is reconstructed from, as α = V/4π (ε − 1). `min_vacuum` refuses the conversion when periodic images are too close, because that error is systematic and looks converged. The training guide documents the sign convention and the vacuum scan for LiF.
* **`mc_mu` must be calibrated.** A guessed chemical potential gives a walk that runs and means nothing. With LiF, the bulk formation energy accepted 1 insertion in 124 attempts and reported nothing wrong. `workflows/lif/validation/gcmc_scan.py` and `gcmc_report.py` show how to scan μ against the insertion energy. The `-9.0` below is a placeholder.

## 3.2 Scaffold the workflow

`scripts/new_workflow.py` writes the standard directory: `run.py`, `training.yaml`, `gap_hypers.yaml` and the READMEs. It then loads the settings back through `config.py`, so a template that has drifted from the code fails at this step rather than in your first dry run.

In [ ]:
sh(f"{PY} scripts/new_workflow.py lico_dipole --species Li C O --force")

## 3.3 Seed structures and the exchanged unit

The seed is **bare geometry**: Li₂CO₃ clusters of 1–4 units with no dipoles on them. When the seed carries no dipoles, the flow first runs VASP on it (`seed structures` → `vasp seed` → `prepare dataset`) and fits iteration 0 to the result. If you already have labelled clusters, put `mu` and `alpha` on them and that step is skipped.

The unit is a planar CO₃ with C–O 1.29 Å. Each Li bridges one O–O edge at Li–O 1.9 Å, which makes the molecule neutral with a 2-fold axis. The clusters are units at random orientations, at least 4 Å apart, rattled slightly. They are starting points for DFT, not minima.

In [ ]:
import numpy as np
from ase import Atoms
from ase.io import write as ase_write
from scipy.spatial.transform import Rotation

DIPOLE_DIR = REPO / "workflows" / "lico_dipole"


def li2co3_unit(r_co=1.29, r_lio=1.90):
    """A neutral Li2CO3 molecule: planar CO3, each Li bridging an O-O edge."""
    angles = np.radians([90.0, 210.0, 330.0])
    o = r_co * np.c_[np.cos(angles), np.sin(angles), np.zeros(3)]
    half_edge = r_co * np.sin(np.radians(60.0))
    r_li = r_co * np.cos(np.radians(60.0)) + np.sqrt(r_lio**2 - half_edge**2)
    li = r_li * np.array([[np.cos(np.radians(a)), np.sin(np.radians(a)), 0.0] for a in (150.0, 30.0)])
    return Atoms("CO3Li2", positions=np.vstack([[0.0, 0.0, 0.0], o, li]))


def cluster(n_units, rng, min_gap=4.0, spread=3.0, rattle=0.05):
    """n randomly oriented units, centres at least min_gap apart."""
    centres = []
    while len(centres) < n_units:
        trial = rng.normal(scale=spread, size=3)
        if all(np.linalg.norm(trial - c) >= min_gap for c in centres):
            centres.append(trial)
    atoms = Atoms()
    for c in centres:
        unit = li2co3_unit()
        unit.positions = Rotation.random(random_state=rng).apply(unit.positions) + c
        atoms += unit
    atoms.positions += rng.normal(scale=rattle, size=atoms.positions.shape)
    return atoms


def seed_set(sizes, per_size, box, seed=0):
    rng = np.random.default_rng(seed)
    frames = []
    for n in sizes:
        for _ in range(per_size):
            atoms = cluster(n, rng)
            atoms.set_cell([box, box, box])
            atoms.center()
            atoms.pbc = False
            frames.append(atoms)
    return frames


BOX = 20.0
unit = li2co3_unit()
unit.info["comment"] = "the neutral Li2CO3 unit the grand-canonical walk inserts and removes"
ase_write(DIPOLE_DIR / "data" / "li2co3_unit.xyz", unit)
seed = seed_set(sizes=[1, 2, 3, 4], per_size=4, box=BOX)
ase_write(DIPOLE_DIR / "data" / "seed_structures.xyz", seed)
print(f"{len(seed)} seed clusters, {min(len(a) for a in seed)}-{max(len(a) for a in seed)} atoms")
print("unit Li-O:", np.round(unit.get_distance(4, 1), 3), "A; net formal charge 2(+1) + 3(-2) + (+4) = 0")

## 3.4 The settings: `training.yaml`

The file overwrites the scaffold's. The comments say why each block is set the way it is. `docs/settings-reference.md` lists every key.

`dataset.box` (20 Å) must exceed twice the descriptor's `rcut_hard` (5 Å in the scaffold's `gap_hypers.yaml`), or the descriptor reaches across the vacuum into the cluster's own image. The scaffold's hyperparameters are a starting point. `workflows/water_ladder/gap_hypers.yaml` explains each value.

In [ ]:
DIPOLE_TRAINING = r"""
# Li2CO3 cluster dipole model, Mode B: the frozen LiCO.gap from Part 2 drives a
# grand-canonical walk; only the dipole model iterates.
#     python workflows/lico_dipole/run.py --dry-run | --stub | (submit)
name: lico_dipole
project: {{PROJECT}}
species_list: [Li, C, O]
iterations: 4

dataset:
  # Bare geometries: the flow computes their dipoles before iteration 0.
  initial: data/seed_structures.xyz
  dipole_key: mu
  polarizability_key: alpha
  dipole_unit: e*angstrom
  polarizability_unit: angstrom^3
  periodic: false
  box: {{BOX}}
  train_fraction: 0.8
  seed: 0

sampling:
  worker: {{TURBOGAP_WORKER}}
  resources: {account: {{ROIHU_ACCOUNT}}, partition: small, nodes: 1,
              ntasks_per_node: 8, time: "02:00:00"}
  method: gcmc
  # The converted .gap from Part 2, on the sampling machine; its gap_files/
  # directory must stay beside it.
  energy_potential: {{LICO_GAP}}
  carry_dipole_model: true

  # Whole neutral units, so every cluster stays neutral.
  mc_species: [Li2CO3]
  mc_molecule_files: [data/li2co3_unit.xyz]
  mc_mu_reference: e0
  # PLACEHOLDER: calibrate against the insertion energy before trusting a run
  # (workflows/lif/validation/gcmc_scan.py shows how).
  mc_mu: [-9.0]
  mc_types: [move, insertion, removal]
  mc_acceptance: [2, 1, 1]
  n_candidates: 100
  discard_initial: 2
  mc:
    mc_nsteps: 400
    mc_move_max: 0.3
    # Below the shortest Li-O bond, so an insertion can bond but not overlap.
    mc_min_dist: 1.5
    # Without it, insertions land in the vacuum of a 20 A box and are rejected.
    mc_max_dist: 4.0
    t_beg: 500.0
    write_xyz: 5

selection:
  method: fps
  n_select: 15
  # Drop collapsed geometries before diversity picks them: FPS prefers
  # whatever is least like the training set, which is exactly those.
  min_separation: 0.9
  max_atoms: 60

vasp:
  worker: {{VASP_WORKER}}
  exec_config: roihu_vasp_config
  molecular: true
  min_vacuum: 8.0
  user_incar_settings:
    LCALCPOL: true
    LEPSILON: true
    ENCUT: 520.0
  # The dispatcher is a few seconds of Python; one structure, one Slurm job,
  # sized by the tiers below (first tier the structure fits).
  resources: {account: {{ROIHU_ACCOUNT}}, partition: small, nodes: 1,
              ntasks_per_node: 1, time: "00:15:00"}
  resource_tiers:
    - max_atoms: 24
      resources: {account: {{ROIHU_ACCOUNT}}, partition: small, nodes: 1,
                  ntasks_per_node: 32, time: "02:00:00"}
    - max_atoms: null
      resources: {account: {{ROIHU_ACCOUNT}}, partition: small, nodes: 1,
                  ntasks_per_node: 128, time: "06:00:00"}
  require_all: false

fit:
  worker: {{FIT_WORKER}}
  resources: {account: {{TRITON_ACCOUNT}}, partition: {{TRITON_PARTITION}}, nodes: 1,
              ntasks: 48, cpus_per_task: 1, time: "12:00:00"}
  hyperparameters_file: gap_hypers.yaml
  mpi_ranks: 48
  num_processes: 1
  default_dipole_sigma: 0.02
  # A pure dipole model has no isolated-atom energy.
  e0: {Li: 0.0, C: 0.0, O: 0.0}
  gap_file_name: lico_dipole.xml

energy_fit:
  # Mode B: the frozen potential drives sampling. The VASP energies and forces
  # still go into the dataset, so turning this on later needs no new DFT.
  enabled: false

validation:
  # A fixed number of iterations. `enabled: true` with `source: generate` adds
  # an independent test walk and stops at a tolerance; see
  # workflows/lif/aims/training.yaml.
  enabled: false
"""
write("workflows/lico_dipole/training.yaml", DIPOLE_TRAINING,
      PROJECT=PROJECT, BOX=BOX, LICO_GAP=LICO_GAP,
      TURBOGAP_WORKER=worker("turbogap"), VASP_WORKER=worker("vasp"),
      FIT_WORKER=worker("gapfit"), ROIHU_ACCOUNT=ROIHU["account"],
      TRITON_ACCOUNT=TRITON["account"], TRITON_PARTITION=TRITON["partition"]);

## 3.5 Dry run, stub, submit

* `--dry-run` prints every job and its worker.
* `--stub` **runs** the whole flow here in seconds. The seed VASP pass, every fit, every GCMC walk and every VASP batch are replaced by stand-ins that return the same shapes, and the merge, selection, payloads and iteration logic between them run for real. It checks the flow, not the physics.
* Then sync and submit.

In [ ]:
DIPOLE_RUN = f"{PY} workflows/lico_dipole/run.py"
sh(DIPOLE_RUN + " --dry-run")

In [ ]:
sh(DIPOLE_RUN + " --stub 2>&1 | tail -n 25")

In [ ]:
sh(f"bash setup/setup_all_machines.sh --config {CONF} --sync-only", act=SUBMIT)
sh(DIPOLE_RUN + f" --project {PROJECT}", act=SUBMIT)

## 3.6 Getting the dipole model out

Every fit job's output carries its potential as a payload in the database's GridFS store, so it can be pulled out long after the run, from any machine. `load=True` is required. Without it the payload comes back as a reference rather than the files.

Compare iterations by the **per-component** test RMSE in e·Å, not by the magnitude error. A model can get every |μ| right while pointing the vectors the wrong way. The held-out set is fixed at the seed split, and that fixed split is what makes iterations comparable at all.

In [ ]:
def fetch_fit(project, flow_name, iteration, out_dir):
    from jobflow_remote import get_jobstore

    from autoplex_soap_turbo.payload import main_file, payload_to_files

    store = get_jobstore(project_name=project)
    store.connect()
    out = store.query_one({"name": f"{flow_name}: fit {iteration}"},
                          properties=["output.potential", "output.test_error"],
                          load=True)["output"]
    payload_to_files(out["potential"], out_dir)
    print(main_file(out["potential"]), "test error:", out["test_error"])


if SUBMIT:
    fetch_fit(PROJECT, "lico_dipole", 3, REPO / "tutorial" / "potentials" / "lico_dipole_3")
else:
    print("[not run] after the flow completes")

---
# Part 4: NMR matching with MADlib and LAMMPS

## 4.1 What MAD does

*Matching with Atomistic Descriptors* adds a bias to the potential energy:

$$E = E_\text{pot} + \tfrac{\gamma}{2}\, D^2\big(S_\text{pred}(\mathbf r),\, S_\text{exp}\big)$$

$S_\text{pred}$ is a spectrum built from per-atom predictions. For NMR, those are the chemical shieldings of each nucleus, broadened by Gaussians onto a ppm grid. $D$ is its distance from the target spectrum. The bias has analytic forces and virial, so MD or minimisation are pushed toward structures whose spectrum matches.

[MADlib](https://github.com/TiganyZ/madlib) is the LAMMPS package that implements this:

| LAMMPS command | what it is |
|---|---|
| `pair_style turbogap` | a turboGAP `.gap` potential read as it stands, with all four terms (soap_turbo, 2b, 3b, core_pot) |
| `compute ID all mad/property/python mad_shiftml ncomp 1 pbc yes` | per-atom shieldings from ShiftML (a metatomic torch model) |
| `compute ID all mad/property/turbogap <file.gap> <property> species …` | per-atom property from a fitted soap_turbo GPR |
| `fix ID group mad/nmr c_ID target <file> range <lo> <hi> <n> …` | the bias. The fix's **group** selects the nucleus |

`fix mad/nmr` keywords: `sigma` sets the broadening in ppm, and the force scales as 1/σ². `gamma` sets the strength in eV, and `gamma g0 g1` ramps it over the run. `every N` applies the bias every N steps. `reference a` / `slope b` map shielding to shift as δ = a − bσ. `convention shift` says the property is already on the target's scale. The fix refuses to run until you give either `reference` or `convention shift`, because fitting shieldings against shifts fits the mirror image. `file`/`write_every` write `# x y_pred y_exp` blocks.

Outputs: `f_ID` is the bias energy, **`f_ID[2]` is the relative error, which is the number to watch**, and `f_ID[3]` is the dissimilarity D². `fix_modify ID energy yes` adds the bias to `pe`.

## 4.2 Building LAMMPS with MADlib

MADlib's installer builds LAMMPS, QUIP and MADlib, plus a Python environment, under `PREFIX`. Each stage is skipped if its output exists, so re-running resumes a failed build. Put `PREFIX` on local or scratch disk: torch alone is hundreds of thousands of files.

The installer does **not** install ShiftML. Add it to the same environment afterwards. Its model weights, about 80 MB, are downloaded once into `~/.cache/shiftml`.

In [ ]:
MADLIB_SRC = Path.home() / "src" / "madlib"            # a checkout of github.com/TiganyZ/madlib
LAMMPS_PREFIX = Path.home() / "work" / "turbogap-lammps"
LMP = LAMMPS_PREFIX / "lammps" / "build" / "lmp"
LMP_VENV = LAMMPS_PREFIX / "venv"

sh(f"git clone https://github.com/TiganyZ/madlib {MADLIB_SRC}",
   act=RUN_LAMMPS and not MADLIB_SRC.exists())
sh(f"MADLIB_SOURCE={MADLIB_SRC} PREFIX={LAMMPS_PREFIX} JOBS=8 "
   f"bash {MADLIB_SRC}/tools/install_turbogap_lammps.sh", act=RUN_LAMMPS)
sh(f"{LMP_VENV}/bin/pip install shiftml", act=RUN_LAMMPS)
# The styles this part needs must be in the binary.
sh(f"{LMP} -h | grep -oE 'mad/(nmr|property/python|property/turbogap)|turbogap' | sort -u",
   act=RUN_LAMMPS)

The installed binary links the system OpenBLAS for soap_turbo, and `libtorch_cpu` also defines `cblas_sgemm_batch`. If OpenBLAS is loaded first, torch's MKL calls go into it and segfault on the first batched GEMM. Preloading `libtorch_cpu.so` avoids this. Note that `compute mad/property/python` takes no arguments, so ShiftML is configured through the environment.

In [ ]:
def nmr_env(types="H C O", model="ShiftML41", device="cpu"):
    """Environment for an lmp that evaluates ShiftML through MADlib."""
    torch_lib = subprocess.run(
        [str(LMP_VENV / "bin" / "python"), "-c",
         "import os, torch; print(os.path.dirname(torch.__file__))"],
        capture_output=True, text=True, check=True).stdout.strip()
    return (f"export PATH={LMP_VENV}/bin:$PATH PYTHONPATH={MADLIB_SRC}/tests:$PYTHONPATH "
            f"LD_PRELOAD={torch_lib}/lib/libtorch_cpu.so "
            f"MADLIB_SHIFTML_TYPES='{types}' MADLIB_SHIFTML_MODEL={model} "
            f"MADLIB_SHIFTML_DEVICE={device} && ")


sh(f"cd {MADLIB_SRC}/tests && LMP={LMP} ./run.sh nmr", act=RUN_LAMMPS)

## 4.3 A worked match: rebuild a crystal from its spectrum

This is the case MADlib's tests verify. Take crystalline succinic acid (C₄H₆O₄, 28 atoms per cell). Its own ShiftML ¹H and ¹³C spectra are the target, with `convention shift`, because both sides are then shieldings from the same model. Then **shuffle the atoms onto each other's sites** and anneal from 1000 K to 300 K with the CHO turboGAP potential, once with the bias and once without. If the bias works, the biased run's `f_nh[2]` and `f_nc[2]` fall well below the control's.

Four steps:

1. predict the pristine crystal's shieldings (`run 0` with a dump of `c_cs`),
2. broaden them into `nmr_H.dat` and `nmr_C.dat`,
3. anneal the shuffled cell with `gamma 0` (the control) and with `gamma > 0`,
4. compare the relative errors and the final spectra.

**Choosing γ.** Run once with `gamma 0` and read `f_ID[3]`, which is D². Then pick γ so that γD²/2 is a small fraction of the cohesive energy. The deck below uses σ = 1.5 ppm (¹H) and 4.0 ppm (¹³C), a 0.5 fs step, and a limited first leg. With σ = 0.5 ppm, a scrambled cell saw forces of 131 eV/Å.

The run is **NVT, not NPT.** The CHO potential has no dispersion, so a molecular crystal under a barostat sublimes.

The CHO turboGAP potential is not part of MADlib. Set `CHO_GAP` to yours. With `nsteps 4000`, each anneal took about 1.5 h on one workstation CPU, and almost all of that is ShiftML.

In [ ]:
NMR_DIR = REPO / "tutorial" / "nmr_succinic"
NMR_DIR.mkdir(parents=True, exist_ok=True)
CHO_GAP = "gap_files/CHO/CHO.gap"   # a turboGAP CHO potential; supply your own

write("tutorial/nmr_succinic/in.predict", r"""
# Per-atom ShiftML shieldings of one structure, as a dump.
units           metal
atom_style      atomic
boundary        p p p
read_data       ${start}
pair_style      zero 3.0
pair_coeff      * *
compute         cs all mad/property/python mad_shiftml ncomp 1 pbc yes
dump            d all custom 1 ${out} id type c_cs
dump_modify     d format float %.8f sort id
run 0
""")

write("tutorial/nmr_succinic/in.anneal", r"""
# Anneal a shuffled cell, biased (gam > 0) or not (gam = 0), towards a target
# 1H + 13C spectrum. Vars: start gam nev seed nsteps tag pot
units           metal
atom_style      atomic
boundary        p p p
read_data       ${start}
pair_style      turbogap e0 H 0.0 e0 C 0.0 e0 O 0.0
pair_coeff      * * ${pot} H C O

group           gH type 1
group           gC type 2
compute         cs all mad/property/python mad_shiftml ncomp 1 pbc yes

# A shuffled cell has wrong-species pairs at bond distances: absorb them first.
velocity        all create 1000.0 ${seed} mom yes rot yes
timestep        0.0005
thermo          200
fix             lim all nve/limit 0.03
run             400
unfix           lim
fix             eq all nvt temp 1000.0 1000.0 0.05
run             600
unfix           eq

fix nh gH mad/nmr c_cs target nmr_H.dat range  5.0  40.0 141 sigma 1.5 &
       convention shift gamma ${gam} every ${nev} file ${tag}_H.spec write_every 500
fix nc gC mad/nmr c_cs target nmr_C.dat range -40.0 200.0 241 sigma 4.0 &
       convention shift gamma ${gam} every ${nev} file ${tag}_C.spec write_every 500
fix_modify      nh energy yes
fix_modify      nc energy yes
thermo_style    custom step temp pe f_nh f_nh[2] f_nc f_nc[2]
thermo_modify   format float %.8g
thermo          500

fix             ann all nvt temp 1000.0 300.0 0.05
run             ${nsteps}
unfix           ann
fix             qch all nvt temp 300.0 10.0 0.05
run             ${nsteps}
unfix           qch

min_style       fire
minimize        0 1e-4 200 2000
write_data      ${tag}_final.data nocoeff
""");

In [ ]:
def read_shieldings(dump, atom_type):
    """Per-atom c_cs of one atom type, from a LAMMPS dump of `id type c_cs`."""
    body = Path(dump).read_text().split("ITEM: ATOMS")[1].splitlines()
    cols = body[0].split()
    rows = [line.split() for line in body[1:] if line.strip()]
    t = np.array([int(r[cols.index("type")]) for r in rows])
    cs = np.array([float(r[cols.index("c_cs")]) for r in rows])
    return cs[t == atom_type]


def broaden(values, lo, hi, n, sigma):
    """The Gaussian sum fix mad/nmr forms on the same grid, L2-normalised."""
    grid = np.linspace(lo, hi, n)
    y = np.exp(-((grid[:, None] - values[None, :]) ** 2) / (2 * sigma**2)).sum(axis=1)
    return grid, y / np.linalg.norm(y)


def write_target(path, grid, y):
    lines = ["# ppm  intensity"] + [f"{x:.8f} {v:.10e}" for x, v in zip(grid, y)]
    Path(path).write_text("\n".join(lines) + "\n")
    print("wrote", path)


def make_targets(dump):
    write_target(NMR_DIR / "nmr_H.dat", *broaden(read_shieldings(dump, 1), 5.0, 40.0, 141, 1.5))
    write_target(NMR_DIR / "nmr_C.dat", *broaden(read_shieldings(dump, 2), -40.0, 200.0, 241, 4.0))


def anneal(tag, gamma, nsteps=4000, every=2, seed=4928):
    env = nmr_env() if RUN_LAMMPS else "<nmr_env> "
    sh(env + f"{LMP} -in in.anneal -var start succinic_shuffled.data -var gam {gamma} "
       f"-var nev {every} -var seed {seed} -var nsteps {nsteps} -var tag {tag} "
       f"-var pot {CHO_GAP} -log {tag}.log", act=RUN_LAMMPS, cwd=NMR_DIR)


if RUN_LAMMPS:
    for name in ("succinic.data", "succinic_shuffled.data"):
        (NMR_DIR / name).unlink(missing_ok=True)
        (NMR_DIR / name).symlink_to(MADLIB_SRC / "tests" / "data" / name)
    sh(nmr_env() + f"{LMP} -in in.predict -var start succinic.data -var out pristine_cs.dump -log none",
       cwd=NMR_DIR)
    make_targets(NMR_DIR / "pristine_cs.dump")
anneal("control", 0.0)
anneal("biased", 20.0)

In [ ]:
def last_spectrum(path):
    """The final block a fix mad/* wrote: its `# step ...` header, then x y_pred y_exp."""
    block = Path(path).read_text().split("# step")[-1].splitlines()
    words = block[0].split()
    header = dict(zip(words[1::2], words[2::2]))
    data = np.array([[float(v) for v in line.split()] for line in block[1:] if line.strip()])
    return float(header["rel_error"]), data[:, 0], data[:, 1], data[:, 2]


if RUN_LAMMPS:
    import matplotlib.pyplot as plt

    for tag in ("control", "biased"):
        err_h = last_spectrum(NMR_DIR / f"{tag}_H.spec")[0]
        err_c = last_spectrum(NMR_DIR / f"{tag}_C.spec")[0]
        print(f"{tag:8s} relative error: 1H {err_h:.3f}, 13C {err_c:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, nucleus in zip(axes, ("H", "C")):
        for tag in ("control", "biased"):
            _, x, pred, target = last_spectrum(NMR_DIR / f"{tag}_{nucleus}.spec")
            ax.plot(x, pred, label=tag)
        ax.plot(x, target, "k--", label="target")
        ax.set_xlabel("shielding (ppm)")
        ax.set_title(f"$^{{{'1' if nucleus == 'H' else '13'}}}${nucleus}")
    axes[0].legend()
    plt.show()

## 4.4 Matching your Li–C–O structures

Point the same deck at the Part 2 potential with `pair_style turbogap` and `pair_coeff * * LiCO.gap Li C O`. Copy the whole `gap_files/` tree with it. For the e0 values, give the isolated-atom energies the RSS run computed, or zero if you only need forces. ¹³C of the carbonate group (≈ 168–171 ppm in Li₂CO₃) and ⁷Li are the natural targets.

**There is one blocker: ShiftML has no lithium.** ShiftML 4.1 is fitted for H, C, N, O, F, Na, Mg, P, S, Cl, K and Ca, and it refuses any structure that contains Li. So for Li–C–O the shielding model has to come from somewhere else. The route MADlib supports is a soap_turbo **local-property GAP**, fitted to per-atom DFT shieldings (GIPAW or similar) of Li–C–O structures. The training structures can come straight from the Part 2 database. MADlib evaluates the model with:

```lammps
compute cs all mad/property/turbogap gap_files/LiCO_cs.gap <property> species Li C O
group   gC type 2
fix     nc gC mad/nmr c_cs target li2co3_C13.dat range 150 190 161 sigma 2.0 &
            reference <a> slope <b> gamma 0.0 every 2 file lico_C.spec write_every 500
```

`LiCO_cs.gap` does not exist yet. Neither this repository nor MADlib fits it. Against a **measured** spectrum, use `reference`/`slope` (δ = a − bσ, calibrated against compounds with known shifts) rather than `convention shift`. Start with `gamma 0` to read off D², as in 4.3.

Any model you can call from Python can also stand in, through `compute mad/property/python <module>` with the `mad_compute`/`mad_adjoint` interface of MADlib's `tests/mad_shiftml.py`.

---
# Where to go next

* **Change a setting**: `docs/settings-reference.md`. `sampling.mc`, `sampling.md`, `vasp.user_incar_settings` and `aims.user_params` pass straight through to turboGAP, VASP or FHI-aims.
* **FHI-aims instead of VASP** for the dipoles: replace the `vasp:` section with an `aims:` section. `workflows/lif/aims/training.yaml` is a configured example, with per-size resource tiers sized by basis-function count.
* **Stop on accuracy, not on a count**: `validation.enabled: true` (`examples/minimal/training_gated.yaml`).
* **Select by committee disagreement**: `selection.method: uncertainty` (`examples/uncertainty/`).
* **A stage or a flow that does not exist**: `docs/custom-workflows.md`, including the four rules. Nothing crosses a job boundary as a path. Every argument must serialise. Strip the model's own outputs. Sync the code to every machine.
* **DFT you already have**: `docs/building-a-training-set.md` turns a tree of finished calculations into one `train.extxyz`.